In [ ]:
import os
import zipfile

# ZIP file location in Google Drive
zip_path = "/content/custwgan_gen2.zip"

# Extract dataset into Colab local storage
extract_path = "/content/new"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
#CNN First One
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # <-- replace with your dataset path
batch_size = 32
img_size = 64   # resize all images to 64x64
num_epochs = 20
learning_rate = 0.001

# ==========================
# DATASET & DATALOADER
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),   # ensure grayscale
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))           # normalize to [-1,1]
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

num_classes = len(train_dataset.classes)
print("Classes:", train_dataset.classes)

# ==========================
# CNN MODEL
# ==========================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (img_size // 8) * (img_size // 8), 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN(num_classes=num_classes).to(device)

# ==========================
# LOSS & OPTIMIZER
# ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# ==========================
# TRAINING
# ==========================
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")

# ==========================
# EVALUATION
# ==========================
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


Classes: ['class_0', 'class_1', 'class_2']
Epoch [1/20], Loss: 0.9873, Train Acc: 45.83%
Epoch [2/20], Loss: 0.6747, Train Acc: 64.04%
Epoch [3/20], Loss: 0.5497, Train Acc: 71.21%
Epoch [4/20], Loss: 0.4679, Train Acc: 74.54%
Epoch [5/20], Loss: 0.4447, Train Acc: 76.58%
Epoch [6/20], Loss: 0.4051, Train Acc: 79.12%
Epoch [7/20], Loss: 0.3798, Train Acc: 81.17%
Epoch [8/20], Loss: 0.3624, Train Acc: 81.96%
Epoch [9/20], Loss: 0.3544, Train Acc: 84.00%
Epoch [10/20], Loss: 0.3211, Train Acc: 84.96%
Epoch [11/20], Loss: 0.2979, Train Acc: 85.38%
Epoch [12/20], Loss: 0.2569, Train Acc: 88.71%
Epoch [13/20], Loss: 0.2744, Train Acc: 87.54%
Epoch [14/20], Loss: 0.2237, Train Acc: 90.00%
Epoch [15/20], Loss: 0.2271, Train Acc: 90.08%
Epoch [16/20], Loss: 0.2121, Train Acc: 90.71%
Epoch [17/20], Loss: 0.1836, Train Acc: 92.33%
Epoch [18/20], Loss: 0.1869, Train Acc: 91.83%
Epoch [19/20], Loss: 0.1491, Train Acc: 93.75%
Epoch [20/20], Loss: 0.1501, Train Acc: 94.17%
Test Accuracy: 85.50%


In [ ]:
#Simple CNN
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from torch.optim.lr_scheduler import StepLR
# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 64     # CNN can work with smaller input
num_classes = 3
num_epochs = 50
lr = 1e-4

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

transform_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform_train)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
#test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# CUSTOM CNN MODEL
# ==========================
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(ImprovedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),   # [B, 32, 64, 64]
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                  # [B, 32, 32, 32]

            nn.Conv2d(32, 64, 3, padding=1),  # [B, 64, 32, 32]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                  # [B, 64, 16, 16]

            nn.Conv2d(64, 128, 3, padding=1), # [B, 128, 16, 16]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),                  # [B, 128, 8, 8]

            nn.Conv2d(128, 256, 3, padding=1),# [B, 256, 8, 8]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),                  # [B, 256, 4, 4]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ==========================
# MODEL, LOSS, OPTIMIZER
# ==========================
model = ImprovedCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)  # reduce LR every 10 epochs

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


Epoch [1/50] Train Loss: 1.0962, Train Acc: 40.75%, Test Acc: 48.17%
Epoch [2/50] Train Loss: 1.0008, Train Acc: 48.38%, Test Acc: 57.33%
Epoch [3/50] Train Loss: 0.9187, Train Acc: 51.83%, Test Acc: 57.67%
Epoch [4/50] Train Loss: 0.8669, Train Acc: 55.25%, Test Acc: 50.33%
Epoch [5/50] Train Loss: 0.8005, Train Acc: 58.62%, Test Acc: 66.67%
Epoch [6/50] Train Loss: 0.7593, Train Acc: 61.46%, Test Acc: 61.33%
Epoch [7/50] Train Loss: 0.6831, Train Acc: 64.67%, Test Acc: 54.50%
Epoch [8/50] Train Loss: 0.6711, Train Acc: 65.17%, Test Acc: 64.33%
Epoch [9/50] Train Loss: 0.6279, Train Acc: 66.71%, Test Acc: 64.33%
Epoch [10/50] Train Loss: 0.5983, Train Acc: 68.21%, Test Acc: 72.67%
Epoch [11/50] Train Loss: 0.5862, Train Acc: 69.17%, Test Acc: 67.50%
Epoch [12/50] Train Loss: 0.5648, Train Acc: 69.62%, Test Acc: 68.17%
Epoch [13/50] Train Loss: 0.5573, Train Acc: 71.21%, Test Acc: 69.50%
Epoch [14/50] Train Loss: 0.5773, Train Acc: 69.96%, Test Acc: 64.17%
Epoch [15/50] Train Loss: 0.5

In [ ]:

import torch
import os

save_dir = "/content/saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "CNN_DayStockGAN.pth")

torch.save(model.state_dict(), model_path)

print(f"✅ Model saved successfully at: {model_path}")


✅ Model saved successfully at: /content/saved_models/CNN_DayStockGAN.pth


In [ ]:
#DeepCNN
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 64     # keep it smaller for custom CNN
num_classes = 3
num_epochs = 50
lr = 1e-3

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),   # grayscale input
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# DEEPER CNN (VGG-style, 5 blocks)
# ==========================
class DeepCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(DeepCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # [B, 64, 32, 32]

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # [B, 128, 16, 16]

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # [B, 256, 8, 8]

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # [B, 512, 4, 4]

            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # [B, 512, 2, 2]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * (img_size // 32) * (img_size // 32), 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ==========================
# MODEL, LOSS, OPTIMIZER
# ==========================
model = DeepCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


Epoch [1/50] Train Loss: 1.1012, Train Acc: 32.54%, Test Acc: 33.33%
Epoch [2/50] Train Loss: 1.0995, Train Acc: 33.04%, Test Acc: 33.33%
Epoch [3/50] Train Loss: 1.0990, Train Acc: 32.33%, Test Acc: 33.33%
Epoch [4/50] Train Loss: 1.0991, Train Acc: 32.67%, Test Acc: 33.33%
Epoch [5/50] Train Loss: 1.0992, Train Acc: 33.00%, Test Acc: 33.33%
Epoch [6/50] Train Loss: 1.0987, Train Acc: 33.50%, Test Acc: 33.33%


KeyboardInterrupt: 

In [ ]:
#VGG16
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 224   # VGG expects 224x224 input
num_classes = 3
num_epochs = 50
lr = 1e-4

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),   # grayscale input
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# VGG16 MODEL (MODIFIED FOR GRAYSCALE + 3 CLASSES)
# ==========================
class VGG16Gray(nn.Module):
    def __init__(self, num_classes=3):
        super(VGG16Gray, self).__init__()
        # Load prebuilt VGG16
        self.vgg16 = models.vgg16(pretrained=True)
        # Change first conv layer (3 channels -> 1 channel)
        self.vgg16.features[0] = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1)
        # Change final classifier for our 3 classes
        self.vgg16.classifier[6] = nn.Linear(4096, num_classes)

    def forward(self, x):
        return self.vgg16(x)

# ==========================
# MODEL, LOSS, OPTIMIZER
# ==========================
model = VGG16Gray(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 238MB/s]


Epoch [1/50] Train Loss: 1.0660, Train Acc: 39.38%, Test Acc: 47.00%
Epoch [2/50] Train Loss: 0.7320, Train Acc: 60.58%, Test Acc: 72.33%
Epoch [3/50] Train Loss: 0.5994, Train Acc: 69.33%, Test Acc: 72.67%
Epoch [4/50] Train Loss: 0.5007, Train Acc: 73.42%, Test Acc: 76.67%
Epoch [5/50] Train Loss: 0.4820, Train Acc: 74.38%, Test Acc: 72.50%
Epoch [6/50] Train Loss: 0.4264, Train Acc: 77.00%, Test Acc: 76.83%
Epoch [7/50] Train Loss: 0.4349, Train Acc: 77.88%, Test Acc: 78.17%
Epoch [8/50] Train Loss: 0.3655, Train Acc: 80.58%, Test Acc: 84.83%
Epoch [9/50] Train Loss: 0.3288, Train Acc: 83.54%, Test Acc: 83.33%
Epoch [10/50] Train Loss: 0.3278, Train Acc: 83.96%, Test Acc: 85.33%
Epoch [11/50] Train Loss: 0.3157, Train Acc: 85.92%, Test Acc: 76.67%
Epoch [12/50] Train Loss: 0.2910, Train Acc: 86.79%, Test Acc: 83.50%
Epoch [13/50] Train Loss: 0.2130, Train Acc: 90.17%, Test Acc: 86.33%
Epoch [14/50] Train Loss: 0.2198, Train Acc: 90.42%, Test Acc: 86.17%
Epoch [15/50] Train Loss: 0.1

In [ ]:

import torch
import os

save_dir = "/content/saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "VGG16_DayStockGAN.pth")

torch.save(model.state_dict(), model_path)

print(f"✅ Model saved successfully at: {model_path}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 64
num_classes = 3
num_epochs = 50
lr = 1e-4

train_acc_list = []
test_acc_list = []
# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Load train and test datasets separately
train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# VGG19 MODEL (with BatchNorm)
# ==========================
class VGG19_Small(nn.Module):
    def __init__(self, num_classes=3):
        super(VGG19_Small, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 2
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 3
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 4
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 5
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512 * (img_size // 32) * (img_size // 32), 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# ==========================
# TRAINING CONFIG
# ==========================
model = VGG19_Small(num_classes).to(device)
criterion = nn.CrossEntropyLoss()

# Larger LR
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Optional: learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")


Epoch [1/50] Train Loss: 1.2713, Train Acc: 33.25%, Test Acc: 33.33%
Epoch [2/50] Train Loss: 1.1204, Train Acc: 32.88%, Test Acc: 34.17%
Epoch [3/50] Train Loss: 1.1010, Train Acc: 35.25%, Test Acc: 34.33%
Epoch [4/50] Train Loss: 1.0899, Train Acc: 35.54%, Test Acc: 39.83%
Epoch [5/50] Train Loss: 1.0225, Train Acc: 44.75%, Test Acc: 53.17%
Epoch [6/50] Train Loss: 0.8139, Train Acc: 58.33%, Test Acc: 62.17%
Epoch [7/50] Train Loss: 0.7089, Train Acc: 62.92%, Test Acc: 68.83%
Epoch [8/50] Train Loss: 0.6278, Train Acc: 67.92%, Test Acc: 68.83%
Epoch [9/50] Train Loss: 0.6106, Train Acc: 69.67%, Test Acc: 64.67%
Epoch [10/50] Train Loss: 0.5632, Train Acc: 70.79%, Test Acc: 49.83%
Epoch [11/50] Train Loss: 0.5572, Train Acc: 71.71%, Test Acc: 68.67%
Epoch [12/50] Train Loss: 0.5265, Train Acc: 73.92%, Test Acc: 67.33%
Epoch [13/50] Train Loss: 0.4522, Train Acc: 78.42%, Test Acc: 69.67%
Epoch [14/50] Train Loss: 0.4611, Train Acc: 77.21%, Test Acc: 72.17%
Epoch [15/50] Train Loss: 0.4

In [ ]:
# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


📌 Evaluation Metrics on Test Set:
Precision: 0.8815
Recall:    0.8317
F1-score:  0.8224

📌 Classification Report:
              precision    recall  f1-score   support

     Class 0       0.99      0.52      0.68       200
     Class 1       0.99      0.99      0.99       200
     Class 2       0.67      0.98      0.80       200

    accuracy                           0.83       600
   macro avg       0.88      0.83      0.82       600
weighted avg       0.88      0.83      0.82       600



#RESNET18

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import os

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 64
num_classes = 3
num_epochs = 50
lr = 1e-4

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # ResNet expects 3 channels
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# RESNET18 MODEL
# ==========================
model = models.resnet18(pretrained=False)   # no ImageNet pretraining
model.fc = nn.Linear(model.fc.in_features, num_classes)  # replace classifier
model = model.to(device)

# ==========================
# LOSS & OPTIMIZER
# ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

    scheduler.step()

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/50] Train Loss: 0.7628, Train Acc: 62.00%, Test Acc: 69.17%
Epoch [2/50] Train Loss: 0.4832, Train Acc: 75.12%, Test Acc: 76.83%
Epoch [3/50] Train Loss: 0.4015, Train Acc: 81.00%, Test Acc: 79.33%
Epoch [4/50] Train Loss: 0.3437, Train Acc: 84.00%, Test Acc: 84.17%
Epoch [5/50] Train Loss: 0.3580, Train Acc: 83.38%, Test Acc: 87.17%
Epoch [6/50] Train Loss: 0.3103, Train Acc: 86.00%, Test Acc: 81.50%
Epoch [7/50] Train Loss: 0.2577, Train Acc: 89.50%, Test Acc: 82.83%
Epoch [8/50] Train Loss: 0.2190, Train Acc: 91.08%, Test Acc: 74.83%
Epoch [9/50] Train Loss: 0.2180, Train Acc: 91.54%, Test Acc: 84.33%
Epoch [10/50] Train Loss: 0.1921, Train Acc: 92.33%, Test Acc: 87.00%
Epoch [11/50] Train Loss: 0.1473, Train Acc: 93.83%, Test Acc: 88.83%
Epoch [12/50] Train Loss: 0.1575, Train Acc: 92.92%, Test Acc: 85.67%
Epoch [13/50] Train Loss: 0.1524, Train Acc: 93.50%, Test Acc: 88.67%
Epoch [14/50] Train Loss: 0.1377, Train Acc: 94.46%, Test Acc: 92.00%
Epoch [15/50] Train Loss: 0.1

In [ ]:

import torch
import os

save_dir = "/content/saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "Resnet18_DayStockGAN.pth")

torch.save(model.state_dict(), model_path)

print(f"✅ Model saved successfully at: {model_path}")


✅ Model saved successfully at: /content/saved_models/Resnet18_DayStockGAN.pth


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import os

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 64
num_classes = 3
num_epochs = 50
lr = 1e-4

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # ResNet expects 3 channels
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# RESNET18 MODEL
# ==========================
model = models.resnet50(pretrained=False)   # no ImageNet pretraining
model.fc = nn.Linear(model.fc.in_features, num_classes)  # replace classifier
model = model.to(device)

# ==========================
# LOSS & OPTIMIZER
# ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

    scheduler.step()

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/50] Train Loss: 1.0427, Train Acc: 53.79%, Test Acc: 53.33%
Epoch [2/50] Train Loss: 0.5299, Train Acc: 75.71%, Test Acc: 67.83%
Epoch [3/50] Train Loss: 0.4300, Train Acc: 80.33%, Test Acc: 81.83%
Epoch [4/50] Train Loss: 0.3307, Train Acc: 85.62%, Test Acc: 76.83%
Epoch [5/50] Train Loss: 0.3301, Train Acc: 85.67%, Test Acc: 84.17%
Epoch [6/50] Train Loss: 0.2887, Train Acc: 88.83%, Test Acc: 78.00%
Epoch [7/50] Train Loss: 0.2411, Train Acc: 89.92%, Test Acc: 75.83%
Epoch [8/50] Train Loss: 0.2485, Train Acc: 90.12%, Test Acc: 86.50%
Epoch [9/50] Train Loss: 0.2076, Train Acc: 91.25%, Test Acc: 90.17%
Epoch [10/50] Train Loss: 0.1504, Train Acc: 94.04%, Test Acc: 90.67%
Epoch [11/50] Train Loss: 0.1654, Train Acc: 93.71%, Test Acc: 91.00%
Epoch [12/50] Train Loss: 0.1732, Train Acc: 93.08%, Test Acc: 78.33%
Epoch [13/50] Train Loss: 0.1916, Train Acc: 92.00%, Test Acc: 91.83%
Epoch [14/50] Train Loss: 0.1539, Train Acc: 93.38%, Test Acc: 89.17%
Epoch [15/50] Train Loss: 0.1

#MobileNetv2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import os

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 64
num_classes = 3
num_epochs = 50
lr = 1e-4

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # MobileNetV2 expects 3 channels
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# MOBILENETV2 MODEL
# ==========================
model = models.mobilenet_v2(pretrained=False)   # set True for ImageNet pretraining
model.classifier[1] = nn.Linear(model.last_channel, num_classes)  # replace final layer
model = model.to(device)

# ==========================
# LOSS & OPTIMIZER
# ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

    scheduler.step()

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/50] Train Loss: 0.9702, Train Acc: 51.29%, Test Acc: 55.83%
Epoch [2/50] Train Loss: 0.5928, Train Acc: 70.50%, Test Acc: 72.50%
Epoch [3/50] Train Loss: 0.4733, Train Acc: 75.96%, Test Acc: 78.00%
Epoch [4/50] Train Loss: 0.3811, Train Acc: 83.50%, Test Acc: 83.17%
Epoch [5/50] Train Loss: 0.3039, Train Acc: 87.25%, Test Acc: 82.33%
Epoch [6/50] Train Loss: 0.2540, Train Acc: 89.75%, Test Acc: 83.83%
Epoch [7/50] Train Loss: 0.1933, Train Acc: 92.04%, Test Acc: 86.83%
Epoch [8/50] Train Loss: 0.1703, Train Acc: 93.46%, Test Acc: 86.50%
Epoch [9/50] Train Loss: 0.1826, Train Acc: 92.67%, Test Acc: 81.33%
Epoch [10/50] Train Loss: 0.1734, Train Acc: 93.92%, Test Acc: 92.33%
Epoch [11/50] Train Loss: 0.0889, Train Acc: 96.92%, Test Acc: 92.33%
Epoch [12/50] Train Loss: 0.1100, Train Acc: 96.04%, Test Acc: 92.00%
Epoch [13/50] Train Loss: 0.1144, Train Acc: 96.25%, Test Acc: 93.83%
Epoch [14/50] Train Loss: 0.0943, Train Acc: 96.42%, Test Acc: 91.83%
Epoch [15/50] Train Loss: 0.0

In [ ]:

import torch
import os

save_dir = "/content/saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "MobileNet_DayStockGAN.pth")

torch.save(model.state_dict(), model_path)

print(f"✅ Model saved successfully at: {model_path}")


✅ Model saved successfully at: /content/saved_models/MobileNet_DayStockGAN.pth


*EfficientNet b0*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import os

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 224   # EfficientNet expects 224x224 (B0)
num_classes = 3
num_epochs = 50
lr = 1e-4

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # EfficientNet expects 3 channels
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# EFFICIENTNET MODEL
# ==========================
model = models.efficientnet_b0(pretrained=True)   # pretrained=True recommended for small datasets
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)  # replace final FC
model = model.to(device)

# ==========================
# LOSS & OPTIMIZER
# ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

    scheduler.step()

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 210MB/s]


Epoch [1/50] Train Loss: 0.6270, Train Acc: 69.33%, Test Acc: 78.33%
Epoch [2/50] Train Loss: 0.3052, Train Acc: 87.00%, Test Acc: 89.83%
Epoch [3/50] Train Loss: 0.1805, Train Acc: 92.71%, Test Acc: 84.83%
Epoch [4/50] Train Loss: 0.1701, Train Acc: 93.42%, Test Acc: 91.33%
Epoch [5/50] Train Loss: 0.0839, Train Acc: 96.46%, Test Acc: 83.67%
Epoch [6/50] Train Loss: 0.0773, Train Acc: 96.88%, Test Acc: 96.33%
Epoch [7/50] Train Loss: 0.0669, Train Acc: 97.62%, Test Acc: 94.33%
Epoch [8/50] Train Loss: 0.0691, Train Acc: 97.58%, Test Acc: 94.33%
Epoch [9/50] Train Loss: 0.0591, Train Acc: 97.71%, Test Acc: 92.33%
Epoch [10/50] Train Loss: 0.0515, Train Acc: 98.12%, Test Acc: 96.67%
Epoch [11/50] Train Loss: 0.0262, Train Acc: 99.08%, Test Acc: 98.00%
Epoch [12/50] Train Loss: 0.0206, Train Acc: 99.38%, Test Acc: 94.17%
Epoch [13/50] Train Loss: 0.0469, Train Acc: 98.54%, Test Acc: 97.50%
Epoch [14/50] Train Loss: 0.0186, Train Acc: 99.33%, Test Acc: 98.83%
Epoch [15/50] Train Loss: 0.0

In [ ]:

import torch
import os

save_dir = "/content/saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "EfficientNetB0_DayStockGAN.pth")

torch.save(model.state_dict(), model_path)

print(f"✅ Model saved successfully at: {model_path}")


✅ Model saved successfully at: /content/saved_models/EfficientNetB0_DayStockGAN.pth


#DensNet121

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import os

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 224    # DenseNet usually works with 224x224
num_classes = 3
num_epochs = 50
lr = 1e-4

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # DenseNet expects 3 channels
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# DENSENET121 MODEL
# ==========================
model = models.densenet121(pretrained=False)   # pretrained=True recommended
model.classifier = nn.Linear(model.classifier.in_features, num_classes)  # replace final layer
model = model.to(device)

# ==========================
# LOSS & OPTIMIZER
# ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

    scheduler.step()

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/50] Train Loss: 0.7293, Train Acc: 61.75%, Test Acc: 69.00%
Epoch [2/50] Train Loss: 0.5187, Train Acc: 71.54%, Test Acc: 74.33%
Epoch [3/50] Train Loss: 0.4579, Train Acc: 75.96%, Test Acc: 72.17%
Epoch [4/50] Train Loss: 0.4616, Train Acc: 76.46%, Test Acc: 65.00%
Epoch [5/50] Train Loss: 0.4003, Train Acc: 80.21%, Test Acc: 83.17%
Epoch [6/50] Train Loss: 0.3603, Train Acc: 82.58%, Test Acc: 71.83%
Epoch [7/50] Train Loss: 0.4044, Train Acc: 82.00%, Test Acc: 82.00%
Epoch [8/50] Train Loss: 0.3368, Train Acc: 84.38%, Test Acc: 79.33%
Epoch [9/50] Train Loss: 0.3065, Train Acc: 86.12%, Test Acc: 79.83%
Epoch [10/50] Train Loss: 0.2665, Train Acc: 88.00%, Test Acc: 86.83%
Epoch [11/50] Train Loss: 0.2948, Train Acc: 87.58%, Test Acc: 76.33%
Epoch [12/50] Train Loss: 0.2463, Train Acc: 89.75%, Test Acc: 78.83%
Epoch [13/50] Train Loss: 0.2368, Train Acc: 89.79%, Test Acc: 83.33%
Epoch [14/50] Train Loss: 0.2341, Train Acc: 90.12%, Test Acc: 81.17%
Epoch [15/50] Train Loss: 0.1

In [ ]:

import torch
import os

save_dir = "/content/saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "DenseNet_DayStockGAN.pth")

torch.save(model.state_dict(), model_path)

print(f"✅ Model saved successfully at: {model_path}")


✅ Model saved successfully at: /content/saved_models/DenseNet_DayStockGAN.pth


In [ ]:
#SqueezNet
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# ==========================
# CONFIG
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/content/new/custwgan_generated"   # should contain "train" and "test" subfolders
batch_size = 32
img_size = 64
num_classes = 3
num_epochs = 50
lr = 1e-3

train_acc_list = []
test_acc_list = []

# ==========================
# TRANSFORMS
# ==========================
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # replicate 1-channel to 3-channel
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=transform)
test_dataset  = datasets.ImageFolder(root=f"{data_dir}/test", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ==========================
# SQUEEZENET MODEL
# ==========================
model = models.squeezenet1_1(pretrained=True)   # can set pretrained=False if you want scratch
model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1), stride=(1,1))
model.num_classes = num_classes
model = model.to(device)

# ==========================
# LOSS & OPTIMIZER
# ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

# ==========================
# TRAINING LOOP
# ==========================
for epoch in range(num_epochs):
    # ---- Training ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_acc_list.append(train_acc)

    # ---- Testing ----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    test_acc = 100 * correct / total
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {running_loss/len(train_loader):.4f}, "
          f"Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%")

    scheduler.step()

# ==========================
# AVERAGE ACCURACY
# ==========================
avg_train_acc = sum(train_acc_list) / len(train_acc_list)
avg_test_acc = sum(test_acc_list) / len(test_acc_list)

print("\n📊 Final Results:")
print(f"Average Train Accuracy: {avg_train_acc:.2f}%")
print(f"Average Test Accuracy: {avg_test_acc:.2f}%")

# ==========================
# FINAL EVALUATION METRICS
# ==========================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📌 Evaluation Metrics on Test Set:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\n📌 Classification Report:")
print(classification_report(y_true, y_pred, target_names=[f"Class {i}" for i in range(num_classes)]))


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SqueezeNet1_1_Weights.IMAGENET1K_V1`. You can also use `weights=SqueezeNet1_1_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/50] Train Loss: 1.1593, Train Acc: 33.58%, Test Acc: 33.33%
Epoch [2/50] Train Loss: 1.0988, Train Acc: 33.29%, Test Acc: 33.33%
Epoch [3/50] Train Loss: 1.0986, Train Acc: 33.29%, Test Acc: 33.33%
Epoch [4/50] Train Loss: 1.0986, Train Acc: 33.29%, Test Acc: 33.33%
Epoch [5/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [6/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [7/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [8/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [9/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [10/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [11/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [12/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [13/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [14/50] Train Loss: 1.0986, Train Acc: 33.33%, Test Acc: 33.33%
Epoch [15/50] Train Loss: 1.0

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/m

In [ ]:

import torch
import os

save_dir = "/content/saved_models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "Squeeznet_DayStockGAN.pth")

torch.save(model.state_dict(), model_path)

print(f"✅ Model saved successfully at: {model_path}")


✅ Model saved successfully at: /content/saved_models/Squeeznet_DayStockGAN.pth
